# BQML Demo — Linear Regression, Logistic Regression, K-Means

In [ ]:
from google.colab import auth, files
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "himanshugcpproject1"
DATASET = "orders"

client = bigquery.Client(project=PROJECT_ID)

def run(sql):
    return client.query(sql).result().to_dataframe()


## Load orders.csv into BigQuery

In [ ]:
uploaded = files.upload()  # choose orders_sample.csv
csv_name = list(uploaded.keys())[0]

df = pd.read_csv(csv_name, parse_dates=["order_date", "order_ts"])

table_id = f"{PROJECT_ID}.{DATASET}.orders_raw"
job = client.load_table_from_dataframe(
    df, table_id,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
)
job.result()
print("loaded", len(df), "rows into", table_id)


## Model 1 — Linear Regression
Goal: predict `amount` (a number) from country, category, and date.

In [ ]:
run(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.amount_predictor`
OPTIONS(model_type='linear_reg', input_label_cols=['amount']) AS
SELECT
  country,
  category,
  EXTRACT(MONTH FROM order_date) AS order_month,
  EXTRACT(DAYOFWEEK FROM order_date) AS order_dow,
  amount
FROM `{PROJECT_ID}.{DATASET}.orders_raw`
""")
print("linear regression model trained")


In [ ]:
print("Evaluation:")
display(run(f"SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET}.amount_predictor`)"))

print("Sample predictions:")
display(run(f"""
SELECT country, category, order_month, order_dow, predicted_amount
FROM ML.PREDICT(MODEL `{PROJECT_ID}.{DATASET}.amount_predictor`,
  (SELECT country, category,
          EXTRACT(MONTH FROM order_date) AS order_month,
          EXTRACT(DAYOFWEEK FROM order_date) AS order_dow
   FROM `{PROJECT_ID}.{DATASET}.orders_raw`
   LIMIT 10))
"""))


## Model 2 — Logistic Regression
Goal: classify whether an order is high-value (amount > 2500), yes or no.

In [ ]:
run(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.high_value_classifier`
OPTIONS(model_type='logistic_reg', input_label_cols=['is_high_value']) AS
SELECT
  country,
  category,
  EXTRACT(MONTH FROM order_date) AS order_month,
  IF(amount > 2500, 1, 0) AS is_high_value
FROM `{PROJECT_ID}.{DATASET}.orders_raw`
""")
print("logistic regression model trained")


In [ ]:
print("Evaluation:")
display(run(f"SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET}.high_value_classifier`)"))

print("Confusion matrix:")
display(run(f"SELECT * FROM ML.CONFUSION_MATRIX(MODEL `{PROJECT_ID}.{DATASET}.high_value_classifier`)"))

print("Sample predictions:")
display(run(f"""
SELECT country, category, order_month,
       predicted_is_high_value, predicted_is_high_value_probs
FROM ML.PREDICT(MODEL `{PROJECT_ID}.{DATASET}.high_value_classifier`,
  (SELECT country, category,
          EXTRACT(MONTH FROM order_date) AS order_month
   FROM `{PROJECT_ID}.{DATASET}.orders_raw`
   LIMIT 10))
"""))


## Model 3 — K-Means
Goal: find natural customer segments from order frequency, spend, and category variety (no label — unsupervised).

In [ ]:
run(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.customer_features` AS
SELECT
  customer_id,
  COUNT(*) AS order_count,
  AVG(amount) AS avg_amount,
  COUNT(DISTINCT category) AS category_variety
FROM `{PROJECT_ID}.{DATASET}.orders_raw`
GROUP BY customer_id
""")

run(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.customer_segments`
OPTIONS(model_type='kmeans', num_clusters=4, standardize_features=TRUE) AS
SELECT order_count, avg_amount, category_variety
FROM `{PROJECT_ID}.{DATASET}.customer_features`
""")
print("k-means model trained")


In [ ]:
print("Evaluation:")
display(run(f"SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET}.customer_segments`)"))

print("Segment profiles (centroids):")
display(run(f"SELECT * FROM ML.CENTROIDS(MODEL `{PROJECT_ID}.{DATASET}.customer_segments`)"))

print("Sample customer -> segment assignments:")
display(run(f"""
SELECT customer_id, order_count, avg_amount, category_variety, CENTROID_ID AS segment
FROM ML.PREDICT(MODEL `{PROJECT_ID}.{DATASET}.customer_segments`,
  TABLE `{PROJECT_ID}.{DATASET}.customer_features`)
LIMIT 10
"""))
